# PyTorch Inference Profiling

Portfolio source extract. See the repository README and PROVENANCE.md for methods, data requirements, original evidence and publication changes. Run from the notebooks/ directory.

In [ ]:
import math
import os
import shutil
import statistics
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")


# Automatic device selection: CUDA -> MPS -> CPU
if torch.cuda.is_available():
    work_device = torch.device("cuda")
    work_dtype = torch.float16
    DEVICE_TYPE = "cuda"
    DEVICE_NAME = torch.cuda.get_device_name(0)

elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    work_device = torch.device("mps")
    work_dtype = torch.float16
    DEVICE_TYPE = "mps"
    DEVICE_NAME = "Apple MPS"

else:
    work_device = torch.device("cpu")
    work_dtype = torch.float32
    DEVICE_TYPE = "cpu"
    DEVICE_NAME = "CPU"

DEVICE_PRECISION = "FP16" if work_dtype == torch.float16 else "FP32"

print("Device:", DEVICE_NAME)
print("Precision:", DEVICE_PRECISION)
print("PyTorch:", torch.__version__)

if DEVICE_TYPE == "cuda":
    print("CUDA:", torch.version.cuda)
    print("Compute capability:", torch.cuda.get_device_capability(0))
    print(
        "GPU memory GiB:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
    )


In [ ]:

def settle_device():
    """Synchronize the selected accelerator. CPU needs no explicit sync."""
    if DEVICE_TYPE == "cuda":
        torch.cuda.synchronize()
    elif DEVICE_TYPE == "mps":
        torch.mps.synchronize()


def empty_device_cache():
    if DEVICE_TYPE == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE_TYPE == "mps" and hasattr(torch.mps, "empty_cache"):
        torch.mps.empty_cache()


def time_call_ms(call, warmup=10, repeats=20, how="median"):
    with torch.no_grad():
        for _ in range(warmup):
            call()
        settle_device()
        times = []
        for _ in range(repeats):
            settle_device()
            start = time.perf_counter()
            call()
            settle_device()
            times.append(time.perf_counter() - start)
    if how == "mean":
        return statistics.mean(times) * 1000
    return statistics.median(times) * 1000


def memory_check_mb(call):
    """Best available memory metric for the selected device."""
    empty_device_cache()
    settle_device()

    if DEVICE_TYPE == "cuda":
        torch.cuda.reset_peak_memory_stats()
        with torch.no_grad():
            out = call()
        settle_device()
        peak_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
        del out
        return peak_mb, "CUDA peak allocated"

    if DEVICE_TYPE == "mps":
        before = torch.mps.current_allocated_memory() if hasattr(torch.mps, "current_allocated_memory") else 0
        with torch.no_grad():
            out = call()
        settle_device()
        after = torch.mps.current_allocated_memory() if hasattr(torch.mps, "current_allocated_memory") else before
        del out
        return max(after - before, 0) / (1024 ** 2), "MPS allocated delta (peak API unavailable)"

    try:
        import psutil
        process = psutil.Process(os.getpid())
        before = process.memory_info().rss
        with torch.no_grad():
            out = call()
        after = process.memory_info().rss
        del out
        return max(after - before, 0) / (1024 ** 2), "CPU RSS delta (peak device memory unavailable)"
    except Exception:
        with torch.no_grad():
            out = call()
        del out
        return np.nan, "CPU peak device memory unavailable"


def count_params(layer):
    return sum(p.numel() for p in layer.parameters())


def bytes_for(dtype):
    return torch.tensor([], dtype=dtype).element_size()


def to_tflops(flops, ms):
    return flops / (ms / 1000) / 1e12


def put_on_device(layer):
    return layer.to(device=work_device, dtype=work_dtype).eval()


In [ ]:
class DepthwiseThenPointwise(nn.Module):
    def __init__(self):
        super().__init__()
        self.depthwise = nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1, groups=64)
        self.pointwise = nn.Conv2d(64, 128, kernel_size=1, stride=1, padding=0)

    def forward(self, x):
        x = self.depthwise(x)
        return self.pointwise(x)


cnn_layers = {
    "Standard 3x3": nn.Conv2d(64, 128, 3, stride=1, padding=1),
    "Depthwise 3x3": nn.Conv2d(64, 64, 3, stride=1, padding=1, groups=64),
    "Pointwise 1x1": nn.Conv2d(64, 128, 1, stride=1, padding=0),
    "Depthwise Separable": DepthwiseThenPointwise(),
}

cnn_macs = {
    "Standard 3x3": 1 * 56 * 56 * 128 * 64 * 3 * 3,
    "Depthwise 3x3": 1 * 56 * 56 * 64 * 3 * 3,
    "Pointwise 1x1": 1 * 56 * 56 * 64 * 128,
    "Depthwise Separable": 1 * 56 * 56 * 64 * 3 * 3 + 1 * 56 * 56 * 64 * 128,
}

cnn_layers = {name: put_on_device(layer) for name, layer in cnn_layers.items()}
cnn_input = torch.randn(1, 64, 56, 56, device=work_device, dtype=work_dtype)


In [ ]:
cnn_rows = []

for name, layer in cnn_layers.items():
    run_layer = lambda layer=layer: layer(cnn_input)
    mem_mb, mem_note = memory_check_mb(run_layer)
    macs = cnn_macs[name]

    cnn_rows.append({
        "Model": name,
        "Parameters": count_params(layer),
        "MACs": macs,
        "FLOPs": 2 * macs,
        "Average Latency ms": time_call_ms(run_layer, warmup=10, repeats=50, how="mean"),
        "GPU Memory MB": mem_mb,
        "Memory Method": mem_note,
    })

cnn_table = pd.DataFrame(cnn_rows)
display(cnn_table)


In [ ]:
class OneBlockTransformer(nn.Module):
    def __init__(self, d=768, h=12, dff=3072):
        super().__init__()
        self.d = d
        self.h = h
        self.dh = d // h

        self.q = nn.Linear(d, d, bias=False)
        self.k = nn.Linear(d, d, bias=False)
        self.v = nn.Linear(d, d, bias=False)
        self.out = nn.Linear(d, d, bias=False)
        self.up = nn.Linear(d, dff, bias=False)
        self.down = nn.Linear(dff, d, bias=False)

    def forward(self, x):
        batch, tokens, width = x.shape

        q = self.q(x).view(batch, tokens, self.h, self.dh).transpose(1, 2)
        k = self.k(x).view(batch, tokens, self.h, self.dh).transpose(1, 2)
        v = self.v(x).view(batch, tokens, self.h, self.dh).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.dh)
        weights = torch.softmax(scores, dim=-1)
        x = torch.matmul(weights, v)

        x = x.transpose(1, 2).contiguous().view(batch, tokens, width)
        x = self.out(x)
        return self.down(torch.relu(self.up(x)))


block = put_on_device(OneBlockTransformer())
block_params = count_params(block)
print("Transformer parameters:", block_params)


In [ ]:
def transformer_macs(tokens, d=768):
    qkv = 3 * tokens * d * d
    qkt = tokens * tokens * d
    av = tokens * tokens * d
    out = tokens * d * d
    mlp = 8 * tokens * d * d
    return qkv + qkt + av + out + mlp


block_rows = []

for tokens in [128, 256, 512, 1024]:
    x = torch.randn(1, tokens, 768, device=work_device, dtype=work_dtype)
    run_block = lambda x=x: block(x)
    mem_mb, mem_note = memory_check_mb(run_block)
    macs = transformer_macs(tokens)

    block_rows.append({
        "Sequence Length": tokens,
        "Parameters": block_params,
        "MACs": macs,
        "FLOPs": 2 * macs,
        "Average Latency ms": time_call_ms(run_block, warmup=10, repeats=50, how="mean"),
        "GPU Memory MB": mem_mb,
        "Memory Method": mem_note,
    })

block_table = pd.DataFrame(block_rows)
display(block_table)


In [ ]:
class DecodeBlock(nn.Module):
    def __init__(self, d=768, h=12):
        super().__init__()
        self.d = d
        self.h = h
        self.dh = d // h

        self.q = nn.Linear(d, d, bias=False)
        self.k = nn.Linear(d, d, bias=False)
        self.v = nn.Linear(d, d, bias=False)
        self.out = nn.Linear(d, d, bias=False)
        self.up = nn.Linear(d, 4 * d, bias=False)
        self.down = nn.Linear(4 * d, d, bias=False)

    def q_for(self, x):
        batch, tokens, width = x.shape
        return self.q(x).view(batch, tokens, self.h, self.dh).transpose(1, 2)

    def kv_for(self, x):
        batch, tokens, width = x.shape
        k = self.k(x).view(batch, tokens, self.h, self.dh).transpose(1, 2)
        v = self.v(x).view(batch, tokens, self.h, self.dh).transpose(1, 2)
        return k, v

    def project(self, x):
        q = self.q_for(x)
        k, v = self.kv_for(x)
        return q, k, v

    def attend(self, q, k, v):
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.dh)
        weights = torch.softmax(scores, dim=-1)
        x = torch.matmul(weights, v)
        batch, heads, tokens, dh = x.shape
        x = x.transpose(1, 2).contiguous().view(batch, tokens, heads * dh)
        return self.out(x)

    def mlp(self, x):
        return self.down(torch.relu(self.up(x)))


decoder = put_on_device(DecodeBlock())


In [ ]:
def decode_without_cache(tokens):
    last = None

    with torch.no_grad():
        for end in range(1, tokens.shape[1] + 1):
            current = tokens[:, end - 1:end, :]
            prefix = tokens[:, :end, :]

            q = decoder.q_for(current)
            k, v = decoder.kv_for(prefix)
            last = decoder.attend(q, k, v)
            last = decoder.mlp(last)

    return last


def decode_with_cache(tokens):
    batch, total_tokens, width = tokens.shape
    k_cache = torch.empty(batch, decoder.h, total_tokens, decoder.dh, device=work_device, dtype=work_dtype)
    v_cache = torch.empty_like(k_cache)
    last = None

    with torch.no_grad():
        for step in range(total_tokens):
            current = tokens[:, step:step + 1, :]
            q = decoder.q_for(current)
            k_new, v_new = decoder.kv_for(current)
            k_cache[:, :, step:step + 1, :] = k_new
            v_cache[:, :, step:step + 1, :] = v_new
            last = decoder.attend(q, k_cache[:, :, :step + 1, :], v_cache[:, :, :step + 1, :])
            last = decoder.mlp(last)

    return last


def kv_cache_mb(tokens, d=768, batch=1):
    return 2 * batch * tokens * d * bytes_for(work_dtype) / (1024 ** 2)


In [ ]:

import subprocess
import threading


def device_power_supported():
    """Power telemetry is available here only for NVIDIA CUDA devices."""
    return DEVICE_TYPE == "cuda" and shutil.which("nvidia-smi") is not None


def read_device_power_w():
    if not device_power_supported():
        return None
    output = subprocess.check_output([
        "nvidia-smi",
        "--query-gpu=power.draw",
        "--format=csv,noheader,nounits",
    ]).decode().strip()
    return float(output.splitlines()[0])


def measure_power_and_energy(fn, num_tokens, sample_interval=0.05, measurement_repeats=3):
    """Measure NVIDIA board power when available. Otherwise return NaN values."""
    if not device_power_supported():
        return {
            "Average Power W": np.nan,
            "Total Energy J": np.nan,
            "Energy per Token J": np.nan,
            "Power Samples": 0,
            "Energy Measurement Time s": np.nan,
            "Power Method": "Power telemetry unavailable on this device",
        }

    power_samples = []
    stop_event = threading.Event()

    def sampler():
        while not stop_event.is_set():
            try:
                value = read_device_power_w()
                if value is not None:
                    power_samples.append(value)
            except Exception:
                pass
            time.sleep(sample_interval)

    settle_device()
    thread = threading.Thread(target=sampler, daemon=True)
    thread.start()
    start = time.perf_counter()
    for _ in range(measurement_repeats):
        fn()
    settle_device()
    end = time.perf_counter()
    stop_event.set()
    thread.join()

    elapsed_s = end - start
    if not power_samples:
        value = read_device_power_w()
        if value is not None:
            power_samples.append(value)

    if not power_samples:
        return {
            "Average Power W": np.nan,
            "Total Energy J": np.nan,
            "Energy per Token J": np.nan,
            "Power Samples": 0,
            "Energy Measurement Time s": elapsed_s,
            "Power Method": "NVIDIA telemetry query returned no samples",
        }

    average_power_w = float(np.mean(power_samples))
    total_energy_j = average_power_w * elapsed_s
    total_generated_tokens = num_tokens * measurement_repeats
    energy_per_token_j = total_energy_j / total_generated_tokens
    return {
        "Average Power W": average_power_w,
        "Total Energy J": total_energy_j,
        "Energy per Token J": energy_per_token_j,
        "Power Samples": len(power_samples),
        "Energy Measurement Time s": elapsed_s,
        "Power Method": "nvidia-smi board power",
    }


cache_rows = []
for tokens in [128, 256, 512, 1024]:
    sample = torch.randn(1, tokens, 768, device=work_device, dtype=work_dtype)
    for mode, run_decode in [
        ("Without Cache", lambda sample=sample: decode_without_cache(sample)),
        ("With Cache", lambda sample=sample: decode_with_cache(sample)),
    ]:
        total_ms = time_call_ms(run_decode, warmup=3, repeats=5, how="mean")
        mem_mb, mem_note = memory_check_mb(run_decode)
        power = measure_power_and_energy(run_decode, num_tokens=tokens)
        cache_rows.append({
            "t": tokens,
            "Mode": mode,
            "Average Latency per Token ms": total_ms / tokens,
            "Total Decoding Latency ms": total_ms,
            "Device Memory MB": mem_mb,
            "Memory Method": mem_note,
            "KV Cache Memory MB": kv_cache_mb(tokens) if mode == "With Cache" else 0,
            **power,
        })

cache_table = pd.DataFrame(cache_rows)
display(cache_table)


In [ ]:

roof_dtype = work_dtype
roof_bytes = bytes_for(roof_dtype)


def measure_matmul_once(size):
    left = torch.randn(size, size, device=work_device, dtype=roof_dtype)
    right = torch.randn(size, size, device=work_device, dtype=roof_dtype)
    result = torch.empty_like(left)

    def run_matmul():
        torch.mm(left, right, out=result)

    latency_ms = time_call_ms(run_matmul, warmup=10, repeats=20, how="median")
    flops = 2 * size ** 3
    throughput = flops / (latency_ms / 1000)
    del left, right, result
    return {
        "N": size,
        "FLOPs": flops,
        "Median Latency ms": latency_ms,
        "Throughput FLOP/s": throughput,
        "Throughput TFLOP/s": throughput / 1e12,
    }


if DEVICE_TYPE == "cpu":
    PEAK_PROBE_SIZES = [512, 1024, 2048]
elif DEVICE_TYPE == "mps":
    PEAK_PROBE_SIZES = [1024, 2048, 4096]
else:
    PEAK_PROBE_SIZES = [1024, 2048, 4096, 8192]

peak_rows = []
for size in PEAK_PROBE_SIZES:
    try:
        peak_rows.append(measure_matmul_once(size))
    except RuntimeError as error:
        print("Skipped peak probe N =", size, "because", str(error).splitlines()[0])

peak_table = pd.DataFrame(peak_rows)
P_peak = peak_table["Throughput FLOP/s"].max()
display(peak_table)
print("Preliminary peak estimate TFLOP/s:", P_peak / 1e12)


In [ ]:
def measure_memory_bandwidth(total_mb=384):
    total_bytes = total_mb * 1024 ** 2
    elements = total_bytes // (3 * roof_bytes)

    a = torch.randn(elements, device=work_device, dtype=roof_dtype)
    b = torch.randn(elements, device=work_device, dtype=roof_dtype)
    c = torch.empty_like(a)

    run_add = lambda: torch.add(a, b, out=c)
    latency_ms = time_call_ms(run_add, warmup=10, repeats=20, how="median")

    bytes_moved = 3 * elements * roof_bytes
    bandwidth = bytes_moved / (latency_ms / 1000)

    del a, b, c

    return {
        "Elements": elements,
        "Bytes Moved": bytes_moved,
        "Median Latency ms": latency_ms,
        "Bandwidth Bytes/s": bandwidth,
        "Bandwidth GB/s": bandwidth / 1e9,
    }


memory_bandwidth = measure_memory_bandwidth()
beta = memory_bandwidth["Bandwidth Bytes/s"]
memory_bandwidth


In [ ]:

if DEVICE_TYPE == "cpu":
    sizes = [64, 128, 256, 512, 1024, 2048]
elif DEVICE_TYPE == "mps":
    sizes = [64, 128, 256, 512, 1024, 2048, 4096]
else:
    sizes = [64, 128, 256, 512, 1024, 2048, 4096, 8192]

results = [measure_matmul_once(size) for size in sizes]

matmul_table = pd.DataFrame()
matmul_table["N"] = sizes
matmul_table["MACs"] = [size ** 3 for size in sizes]
matmul_table["FLOPs"] = [result["FLOPs"] for result in results]
matmul_table["Bytes Moved"] = [roof_bytes * 3 * size ** 2 for size in sizes]
matmul_table["Arithmetic Intensity"] = matmul_table["FLOPs"] / matmul_table["Bytes Moved"]
matmul_table["Median Latency ms"] = [result["Median Latency ms"] for result in results]
matmul_table["Achieved TFLOP/s"] = [result["Throughput FLOP/s"] / 1e12 for result in results]

best_sweep = matmul_table["Achieved TFLOP/s"].max() * 1e12
P_peak = max(P_peak, best_sweep)
I_star = P_peak / beta
matmul_table["Fraction of Measured Peak"] = matmul_table["Achieved TFLOP/s"] / (P_peak / 1e12)

display(matmul_table)
print("Final measured peak compute TFLOP/s:", P_peak / 1e12)
print("Final transition intensity FLOPs/byte:", I_star)


In [ ]:
plt.figure(figsize=(8, 5))

x = matmul_table["Arithmetic Intensity"]
y = matmul_table["Achieved TFLOP/s"]

plt.loglog(x, y, marker="o", label="Measured matmul")

roof_x = np.logspace(
    math.log10(max(0.1, x.min() / 2)),
    math.log10(x.max() * 2),
    200,
)
roof_y = np.minimum(P_peak / 1e12, beta * roof_x / 1e12)

plt.loglog(roof_x, roof_y, label="Measured roofline")
plt.xlabel("Arithmetic Intensity FLOPs/byte")
plt.ylabel("Throughput TFLOP/s")
plt.title("Matrix Multiplication Roofline")
plt.grid(True, which="both")
plt.legend()
plt.show()


In [ ]:
RTX_3090_REFERENCE_TFLOPS = 71.0
RTX_3090_REFERENCE_LABEL = "RTX 3090 dense FP16 Tensor Core peak with FP32 accumulation"
RTX_3090_REFERENCE_SOURCE = "https://www.nvidia.com/content/PDF/nvidia-ampere-ga-102-gpu-architecture-whitepaper-v2.pdf"

DATASHEET_PEAK_TFLOPS = RTX_3090_REFERENCE_TFLOPS

highest_sweep_tflops = matmul_table["Achieved TFLOP/s"].max()
measured_peak_tflops = P_peak / 1e12
highest_intensity_row = matmul_table.loc[matmul_table["Arithmetic Intensity"].idxmax()]
highest_intensity_tflops = highest_intensity_row["Achieved TFLOP/s"]

print("Device:", DEVICE_NAME)
print("Precision:", DEVICE_PRECISION)
print("Measured peak TFLOP/s:", measured_peak_tflops)
print(RTX_3090_REFERENCE_LABEL + ":", DATASHEET_PEAK_TFLOPS, "TFLOP/s")
print("Reference:", RTX_3090_REFERENCE_SOURCE)
print("Highest sweep throughput TFLOP/s:", highest_sweep_tflops)
print("Highest sweep throughput / measured peak:", highest_sweep_tflops / measured_peak_tflops)
print("Highest sweep throughput / reference peak:", highest_sweep_tflops / DATASHEET_PEAK_TFLOPS)
print("Highest-intensity configuration N:", int(highest_intensity_row["N"]))
print("Highest-intensity throughput / measured peak:", highest_intensity_tflops / measured_peak_tflops)
print("Highest-intensity throughput / reference peak:", highest_intensity_tflops / DATASHEET_PEAK_TFLOPS)


In [ ]:
batch_rows = []

Cin = 64
Cout = 128
H = 56
W = 56
K = 3

conv_for_batch = put_on_device(nn.Conv2d(Cin, Cout, K, stride=1, padding=1))
weight_elements = Cout * Cin * K * K
weight_bytes = weight_elements * roof_bytes

for batch in [1, 2, 4, 8, 16, 32, 64, 128]:
    try:
        x = torch.randn(batch, Cin, H, W, device=work_device, dtype=roof_dtype)
        run_conv = lambda x=x: conv_for_batch(x)
        latency_ms = time_call_ms(run_conv, warmup=10, repeats=20, how="median")

        macs = batch * H * W * Cout * Cin * K * K
        flops = 2 * macs
        input_elements = batch * Cin * H * W
        output_elements = batch * Cout * H * W
        bytes_moved = roof_bytes * (input_elements + weight_elements + output_elements)
        throughput = flops / (latency_ms / 1000)

        batch_rows.append({
            "Batch": batch,
            "MACs": macs,
            "FLOPs": flops,
            "Bytes Moved": bytes_moved,
            "Arithmetic Intensity": flops / bytes_moved,
            "Median Latency ms": latency_ms,
            "Achieved TFLOP/s": throughput / 1e12,
            "Weight Traffic Fraction": weight_bytes / bytes_moved,
        })

        del x
    except RuntimeError as error:
        print("Stopped at batch =", batch, "because", str(error).splitlines()[0])
        break

batch_table = pd.DataFrame(batch_rows)
display(batch_table)


In [ ]:
I_limit = (
    2 * H * W * Cout * Cin * K * K
) / (
    roof_bytes * (Cin * H * W + Cout * H * W)
)

batch_one = batch_table.loc[batch_table["Batch"] == 1].iloc[0]

print("Measured I range:", batch_table["Arithmetic Intensity"].min(), "to", batch_table["Arithmetic Intensity"].max())
print("Analytical limit of I(B):", I_limit)
print("Weight traffic fraction at B = 1:", batch_one["Weight Traffic Fraction"])
print("Largest possible I increase from B = 1:", I_limit / batch_one["Arithmetic Intensity"])


In [ ]:
batch_intensity = batch_table["Arithmetic Intensity"]
batch_throughput = batch_table["Achieved TFLOP/s"]

roof_x = np.logspace(
    math.log10(batch_intensity.min() * 0.95),
    math.log10(batch_intensity.max() * 1.05),
    100,
)
roof_y = np.minimum(P_peak / 1e12, beta * roof_x / 1e12)

plt.figure(figsize=(8, 5))
plt.loglog(batch_intensity, batch_throughput, marker="o", label="Measured convolution")
plt.loglog(roof_x, roof_y, label="Measured roofline")
plt.xlabel("Arithmetic Intensity FLOPs/byte")
plt.ylabel("Achieved Throughput TFLOP/s")
plt.title("Standard Convolution Batch Roofline")
plt.grid(True, which="both")
plt.legend()
plt.show()


In [ ]:
conv_shapes = [
    (16, 16, 1),
    (16, 32, 3),
    (32, 64, 1),
    (32, 64, 3),
    (64, 64, 3),
    (64, 128, 1),
    (64, 128, 3),
    (128, 128, 3),
    (128, 256, 1),
    (128, 256, 3),
    (256, 256, 3),
    (256, 512, 1),
    (256, 512, 3),
    (256, 512, 5),
]

wide_rows = []

for Cin, Cout, K in conv_shapes:
    try:
        layer = put_on_device(nn.Conv2d(Cin, Cout, K, stride=1, padding=K // 2))
        x = torch.randn(1, Cin, 56, 56, device=work_device, dtype=roof_dtype)
        run_conv = lambda layer=layer, x=x: layer(x)
        latency_ms = time_call_ms(run_conv, warmup=10, repeats=20, how="median")

        macs = 56 * 56 * Cout * Cin * K * K
        flops = 2 * macs
        input_elements = Cin * 56 * 56
        weight_elements = Cout * Cin * K * K
        output_elements = Cout * 56 * 56
        bytes_moved = roof_bytes * (input_elements + weight_elements + output_elements)
        throughput = flops / (latency_ms / 1000)

        wide_rows.append({
            "Cin": Cin,
            "Cout": Cout,
            "K": K,
            "MACs": macs,
            "FLOPs": flops,
            "Bytes Moved": bytes_moved,
            "Arithmetic Intensity": flops / bytes_moved,
            "Median Latency ms": latency_ms,
            "Achieved TFLOP/s": throughput / 1e12,
        })

        del layer, x
    except RuntimeError as error:
        print("Skipped", (Cin, Cout, K), "because", str(error).splitlines()[0])

wide_table = pd.DataFrame(wide_rows)
display(wide_table)


In [ ]:
wide_intensity = wide_table["Arithmetic Intensity"]
wide_throughput = wide_table["Achieved TFLOP/s"]

roof_x = np.logspace(
    math.log10(wide_intensity.min() * 0.8),
    math.log10(wide_intensity.max() * 1.2),
    200,
)
roof_y = np.minimum(P_peak / 1e12, beta * roof_x / 1e12)

plt.figure(figsize=(8, 5))
plt.loglog(
    wide_intensity,
    wide_throughput,
    marker="o",
    linestyle="none",
    label="Measured convolution",
)
plt.loglog(roof_x, roof_y, label="Measured roofline")

for _, row in wide_table.iterrows():
    label = f'{int(row["Cin"])}-{int(row["Cout"])}-K{int(row["K"])}'
    plt.annotate(label, (row["Arithmetic Intensity"], row["Achieved TFLOP/s"]), fontsize=8)

plt.xlabel("Arithmetic Intensity FLOPs/byte")
plt.ylabel("Achieved Throughput TFLOP/s")
plt.title("Wider Convolution Roofline")
plt.grid(True, which="both")
plt.legend()
plt.show()


In [ ]:
def prefill_numbers(tokens, d=768, h=12, b=roof_bytes):
    macs = transformer_macs(tokens, d)
    flops = 2 * macs

    weight_elements = 12 * d * d
    activation_elements = (
        tokens * d
        + 3 * tokens * d
        + h * tokens * tokens
        + h * tokens * tokens
        + tokens * d
        + tokens * 4 * d
        + tokens * d
    )

    bytes_moved = b * (weight_elements + activation_elements)
    return macs, flops, bytes_moved, flops / bytes_moved


def decode_numbers(tokens, d=768, b=roof_bytes):
    qkv = 3 * d * d
    attention = 2 * tokens * d
    out = d * d
    mlp = 8 * d * d

    macs = qkv + attention + out + mlp
    flops = 2 * macs
    weight_elements = 12 * d * d
    current_token_elements = d + 3 * d + d + 4 * d + d
    kv_cache_elements = 2 * tokens * d

    bytes_moved = b * (weight_elements + current_token_elements + kv_cache_elements)
    return macs, flops, bytes_moved, flops / bytes_moved


In [ ]:
prefill_rows = []
decode_rows = []

for tokens in [128, 256, 512, 1024]:
    x = torch.randn(1, tokens, 768, device=work_device, dtype=work_dtype)
    run_prefill = lambda x=x: block(x)
    prefill_ms = time_call_ms(run_prefill, warmup=10, repeats=20, how="median")
    macs, flops, bytes_moved, intensity = prefill_numbers(tokens)

    prefill_rows.append({
        "Sequence Length": tokens,
        "Phase": "Prefill",
        "MACs": macs,
        "FLOPs": flops,
        "Bytes Moved": bytes_moved,
        "Arithmetic Intensity": intensity,
        "Median Latency ms": prefill_ms,
        "Achieved TFLOP/s": to_tflops(flops, prefill_ms),
    })

    one_token = torch.randn(1, 1, 768, device=work_device, dtype=work_dtype)
    k_all = torch.randn(1, 12, tokens + 1, 64, device=work_device, dtype=work_dtype)
    v_all = torch.randn(1, 12, tokens + 1, 64, device=work_device, dtype=work_dtype)

    def run_decode_step():
        q = decoder.q_for(one_token)
        k_new, v_new = decoder.kv_for(one_token)
        k_all[:, :, -1:, :].copy_(k_new)
        v_all[:, :, -1:, :].copy_(v_new)
        out = decoder.attend(q, k_all, v_all)
        return decoder.mlp(out)

    decode_ms = time_call_ms(run_decode_step, warmup=10, repeats=20, how="median")
    macs, flops, bytes_moved, intensity = decode_numbers(tokens)

    decode_rows.append({
        "Sequence Length": tokens,
        "Phase": "Decode",
        "MACs": macs,
        "FLOPs": flops,
        "Bytes Moved": bytes_moved,
        "Arithmetic Intensity": intensity,
        "Median Latency ms": decode_ms,
        "Achieved TFLOP/s": to_tflops(flops, decode_ms),
        "KV Cache MB": kv_cache_mb(tokens),
    })

prefill_table = pd.DataFrame(prefill_rows)
decode_table = pd.DataFrame(decode_rows)

print("Prefill")
display(prefill_table)

print("Decode")
display(decode_table)


In [ ]:
plt.figure(figsize=(8, 5))

plt.loglog(prefill_table["Arithmetic Intensity"], prefill_table["Achieved TFLOP/s"], marker="o", label="Prefill")
plt.loglog(decode_table["Arithmetic Intensity"], decode_table["Achieved TFLOP/s"], marker="o", label="Decode")

all_intensity = pd.concat([prefill_table["Arithmetic Intensity"], decode_table["Arithmetic Intensity"]])
roof_x = np.logspace(
    math.log10(max(0.01, all_intensity.min() / 2)),
    math.log10(all_intensity.max() * 2),
    200,
)
roof_y = np.minimum(P_peak / 1e12, beta * roof_x / 1e12)

plt.loglog(roof_x, roof_y, label="Measured roofline")
plt.xlabel("Arithmetic Intensity FLOPs/byte")
plt.ylabel("Achieved Throughput TFLOP/s")
plt.title("Transformer Prefill and Decode Roofline")
plt.grid(True, which="both")
plt.legend()
plt.show()


In [ ]:
def highest_intensity_point(table, workload, setting):
    row = table.loc[table["Arithmetic Intensity"].idxmax()]
    return {
        "Workload": workload,
        "Configuration": setting(row),
        "Arithmetic Intensity": row["Arithmetic Intensity"],
        "Achieved TFLOP/s": row["Achieved TFLOP/s"],
    }


highest_points = pd.DataFrame([
    highest_intensity_point(matmul_table, "Matrix multiplication", lambda row: f'N={int(row["N"])}'),
    highest_intensity_point(batch_table, "Convolution batch", lambda row: f'B={int(row["Batch"])}'),
    highest_intensity_point(
        wide_table,
        "Wider convolution",
        lambda row: f'Cin={int(row["Cin"])}, Cout={int(row["Cout"])}, K={int(row["K"])}',
    ),
    highest_intensity_point(prefill_table, "Transformer prefill", lambda row: f'N={int(row["Sequence Length"])}'),
    highest_intensity_point(decode_table, "Transformer decode", lambda row: f'N={int(row["Sequence Length"])}'),
])

highest_points["Fraction of Measured Peak"] = highest_points["Achieved TFLOP/s"] / (P_peak / 1e12)
highest_points["Fraction of Datasheet Peak"] = (
    highest_points["Achieved TFLOP/s"] / DATASHEET_PEAK_TFLOPS
    if DATASHEET_PEAK_TFLOPS is not None
    else np.nan
)

overall_highest = highest_points.loc[highest_points["Arithmetic Intensity"].idxmax()]

print("Highest-intensity point from each sweep")
display(highest_points)
print("Overall highest-intensity workload:", overall_highest["Workload"])
print("Configuration:", overall_highest["Configuration"])
print("Arithmetic intensity FLOPs/byte:", overall_highest["Arithmetic Intensity"])
print("Fraction of measured peak:", overall_highest["Fraction of Measured Peak"])
print("Fraction of datasheet peak:", overall_highest["Fraction of Datasheet Peak"])

summary = {
    "Measured Peak Compute TFLOP/s": P_peak / 1e12,
    "Measured Memory Bandwidth GB/s": beta / 1e9,
    "Transition Intensity FLOPs/byte": I_star,
    "Matmul Min I": matmul_table["Arithmetic Intensity"].min(),
    "Matmul Max I": matmul_table["Arithmetic Intensity"].max(),
    "Conv Batch Min I": batch_table["Arithmetic Intensity"].min(),
    "Conv Batch Max I": batch_table["Arithmetic Intensity"].max(),
    "Wide Conv Min I": wide_table["Arithmetic Intensity"].min(),
    "Wide Conv Max I": wide_table["Arithmetic Intensity"].max(),
    "Prefill Min I": prefill_table["Arithmetic Intensity"].min(),
    "Prefill Max I": prefill_table["Arithmetic Intensity"].max(),
    "Decode Min I": decode_table["Arithmetic Intensity"].min(),
    "Decode Max I": decode_table["Arithmetic Intensity"].max(),
}

summary_table = pd.DataFrame(summary.items(), columns=["Metric", "Value"])
display(summary_table)


In [ ]:
def roofline_region(intensity):
    if intensity < I_star:
        return "Memory bound"
    return "Compute bound or near compute bound"


print("Matrix multiplication")
for _, row in matmul_table.iterrows():
    print(int(row["N"]), roofline_region(row["Arithmetic Intensity"]))

print("\nConvolution batch sweep")
for _, row in batch_table.iterrows():
    print(int(row["Batch"]), roofline_region(row["Arithmetic Intensity"]))

print("\nWider convolution sweep")
for _, row in wide_table.iterrows():
    label = f'{int(row["Cin"])}-{int(row["Cout"])}-K{int(row["K"])}'
    print(label, roofline_region(row["Arithmetic Intensity"]))

print("\nTransformer prefill")
for _, row in prefill_table.iterrows():
    print(int(row["Sequence Length"]), roofline_region(row["Arithmetic Intensity"]))

print("\nTransformer decode")
for _, row in decode_table.iterrows():
    print(int(row["Sequence Length"]), roofline_region(row["Arithmetic Intensity"]))
